<a href="https://colab.research.google.com/github/fbeilstein/bioinformatics/blob/master/practice_01_fastp_mash.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required dependencies
!apt-get -y install mash
!wget http://opendata.pku.edu.cn/dataset.xhtml?persistentId=doi:10.18170/DVN/PCEF71/FASTP -O fastp
!chmod a+x ./fastp

# 1. Install SRA Toolkit for downloading sequencing data
!apt-get -y install sra-toolkit

# 2. Download Reference Genomes (NCBI RefSeq)
# E. coli K-12 MG1655
!wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/005/845/GCF_000005845.2_ASM584v2/GCF_000005845.2_ASM584v2_genomic.fna.gz -O ecoli_ref.fna.gz

# Salmonella Typhimurium LT2
!wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/006/945/GCF_000006945.2_ASM694v2/GCF_000006945.2_ASM694v2_genomic.fna.gz -O salmonella_ref.fna.gz

# 3. Download Raw Query Reads (E. coli Illumina paired-end dataset)
# The -X 25000 flag strictly limits the download to the first 25,000 read pairs (approx. 10-15 MB).
# The --split-files flag generates two files: SRR10971019_1.fastq and SRR10971019_2.fastq.
!fastq-dump -X 25000 --split-files SRR10971019

**debug**

In [12]:
# 1. Download the paired-end reads directly via HTTPS
!wget https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR258/003/SRR2584863/SRR2584863_1.fastq.gz -O raw_reads_1.fastq.gz
!wget https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR258/003/SRR2584863/SRR2584863_2.fastq.gz -O raw_reads_2.fastq.gz

# 2. Run fastp on the downloaded compressed files
!./fastp -i raw_reads_1.fastq.gz -I raw_reads_2.fastq.gz -o clean_reads_1.fastq -O clean_reads_2.fastq -h report.html

# 3. Sketch the cleaned forward reads
!mash sketch -k 21 -m 2 clean_reads_1.fastq

# 4. Calculate distances against the references
!mash dist ecoli_ref.fna.gz.msh clean_reads_1.fastq.msh
!mash dist salmonella_ref.fna.gz.msh clean_reads_1.fastq.msh

--2026-09-16 17:44:24--  https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR258/003/SRR2584863/SRR2584863_1.fastq.gz
Resolving ftp.sra.ebi.ac.uk (ftp.sra.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.sra.ebi.ac.uk (ftp.sra.ebi.ac.uk)|193.62.193.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 183292913 (175M) [application/x-gzip]
Saving to: ‘raw_reads_1.fastq.gz’

raw_reads_1.fastq.g 100%[===================>] 174.80M  2.26MB/s    in 26s     

2026-09-16 17:44:54 (6.61 MB/s) - ‘raw_reads_1.fastq.gz’ saved [183292913/183292913]

--2026-09-16 17:44:54--  https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR258/003/SRR2584863/SRR2584863_2.fastq.gz
Resolving ftp.sra.ebi.ac.uk (ftp.sra.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.sra.ebi.ac.uk (ftp.sra.ebi.ac.uk)|193.62.193.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 191090316 (182M) [application/x-gzip]
Saving to: ‘raw_reads_2.fastq.gz’

raw_reads_2.fastq.g 100%[===================>] 182.24M  17.6M

In [14]:
# 1. Download the working fastp binary
!wget http://opengene.org/fastp/fastp -O fastp
!chmod a+x ./fastp

# 2. Run fastp on your successfully downloaded raw reads
!./fastp -i raw_reads_1.fastq.gz -I raw_reads_2.fastq.gz -o clean_reads_1.fastq -O clean_reads_2.fastq -h report.html

# 3. Sketch the cleaned forward reads
!mash sketch -k 21 -m 2 clean_reads_1.fastq

# 4. Calculate distances against the references
!mash dist ecoli_ref.fna.gz.msh clean_reads_1.fastq.msh
!mash dist salmonella_ref.fna.gz.msh clean_reads_1.fastq.msh

--2026-09-16 17:47:36--  http://opengene.org/fastp/fastp
Resolving opengene.org (opengene.org)... 8.210.133.117
Connecting to opengene.org (opengene.org)|8.210.133.117|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://opengene.org/fastp/fastp [following]
--2026-09-16 17:47:36--  https://opengene.org/fastp/fastp
Connecting to opengene.org (opengene.org)|8.210.133.117|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13785864 (13M) [application/octet-stream]
Saving to: ‘fastp’

fastp               100%[===================>]  13.15M  5.28MB/s    in 2.5s    

2026-09-16 17:47:40 (5.28 MB/s) - ‘fastp’ saved [13785864/13785864]

Read1 before filtering:
total reads: 1553259
total bases: 232988850
Q20 bases: 222136776(95.3422%)
Q30 bases: 208265689(89.3887%)
Q40 bases: 44466587(19.0853%)

Read2 before filtering:
total reads: 1553259
total bases: 232988850
Q20 bases: 193356778(82.9897%)
Q30 bases: 172128273(73.8783%)
Q40 ba

**Part 1:** Quality Control with fastp

Raw sequencing data contains adapter sequences and low-quality base calls (Phred score $Q < 20$). We use `fastp` to trim adapters via overlap analysis and filter poor-quality reads before downstream processing.

In [6]:
# Run fastp to clean the raw sequencing data
!./fastp -i SRR10971019_1.fastq -I SRR10971019_2.fastq -o clean_reads_1.fastq -O clean_reads_2.fastq -h report.html

**Part 2:** MinHash Sketching with mash

Exact alignment is computationally expensive. We reduce the reference genomes and our cleaned reads into compressed MinHash sketches ($k=21$) to rapidly estimate the Jaccard similarity and Mash distance.

In [7]:
# Sketch the reference genomes
!mash sketch -k 21 ecoli_ref.fna.gz
!mash sketch -k 21 salmonella_ref.fna.gz

# Sketch the cleaned forward reads (filtering single-occurrence k-mers with -m 2)
!mash sketch -k 21 -m 2 clean_reads_1.fastq

# Calculate distance between the read sketch and both references
!mash dist ecoli_ref.fna.gz.msh clean_reads_1.fastq.msh
!mash dist salmonella_ref.fna.gz.msh clean_reads_1.fastq.msh


Sketching ecoli_ref.fna.gz...
Writing to ecoli_ref.fna.gz.msh...
Sketching salmonella_ref.fna.gz...
Writing to salmonella_ref.fna.gz.msh...
ERROR: could not open clean_reads_1.fastq
ERROR: could not open "clean_reads_1.fastq.msh" for reading.
ERROR: could not open "clean_reads_1.fastq.msh" for reading.
